# 第 13 章习题与解答

## Exercise 13.1

**题目**:GAE(Generalized Advantage Estimation)为什么比直接使用 reward 作为 advantage 更好?

<details><summary><b>参考答案</b></summary>

直接用 reward 作为 advantage 的问题是**方差太大**。某些 token 可能恰好碰上高 reward,但并不真的是「好动作」—— 可能只是运气好。

GAE 通过 Critic 的 V(s) 提供 baseline:

$$\hat{A}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l (r_{t+l} + \gamma V(s_{t+l+1}) - V(s_{t+l}))$$

- $V(s_t)$ 估计「在状态 $s_t$ 的平均期望回报」
- $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ 是 TD 误差 —— 「实际比预期好多少」
- 方差显著降低,因为减去了 baseline

> $\lambda$ 控制 bias-variance tradeoff:$\lambda=0$ 纯 TD(低方差高偏差),$\lambda=1$ 纯 Monte Carlo(高方差低偏差)。通常 $\lambda=0.95$。

</details>

## Exercise 13.2

**题目**:GRPO 去掉了 Critic 模型。为什么组归一化能替代 Critic 的作用?

<details><summary><b>参考答案</b></summary>

Critic 的核心作用是提供 **baseline** —— 「平均能拿多少分」。优势 = 实际分数 − baseline。

GRPO 对每个 prompt 生成 N 个回复,组均值就是一个**天然的 baseline**:

$$A_i = \frac{r_i - \bar{r}}{\sigma_r}$$

- 如果一个回复的 reward 高于组均值 → 正优势(鼓励)
- 如果低于组均值 → 负优势(抑制)

这完全替代了 Critic:
- Critic 需要**额外训练**一个价值网络
- 组均值**不需要训练**,只需要多次采样

> **代价**:GRPO 每个提示需要 N 次生成(minimind N=6),推理成本是 PPO 的 ~6 倍。但省掉了 Critic 的训练成本和显存。

</details>

## Exercise 13.3

**题目**:CISPO 的 `clamp(ratio, max=ε_high)` 与 PPO 的 `clip(ratio, 1-ε, 1+ε)` 有什么本质区别?为什么 CISPO 更稳定?

<details><summary><b>参考答案</b></summary>

**PPO 的 clip**:双向裁剪,比率 r 被限制在 [1-ε, 1+ε]:

```python
surr = min(r * A, clip(r, 1-0.2, 1+0.2) * A)
```

当 $A > 0$(好动作)且 $r > 1+ε$ 时,梯度被截断 —— 模型**无法进一步强化**已经很好的动作。

**CISPO 的 clamp**:只裁剪上界,不裁剪下界:

```python
surr = clamp(r, max=5.0) * A * log_prob
```

- $A > 0$ 且 $r < 5.0$:梯度正常流动(允许大正向更新)
- $A > 0$ 且 $r > 5.0$:截断(防止极端偏离)
- $A < 0$:不裁剪(正常抑制坏动作)

**本质区别**:PPO 限制了「好动作的强化幅度」,CISPO 只限制「极端偏离」。在训练早期,好的探索动作需要快速强化,CISPO 允许这一点而 PPO 不行 —— 这就是 CISPO 更稳定的原因。

</details>